### Predict whether a customer will churn during a crisis based on pre-crisis behavior.

In [0]:
# Import Libraries

import mlflow
import mlflow.spark

from pyspark.sql import functions as F
from pyspark.sql.types import IntegerType

from pyspark.ml.feature import VectorAssembler, StringIndexer
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import BinaryClassificationEvaluator
from pyspark.ml import Pipeline

2. Label_clasification: 

churn = 1 → customer stopped ordering during crisis
churn = 0 → customer continued ordering

In [0]:
# Load Silver table
orders_df = spark.table("uc_quickbite.silver_transform.orders_enriched")

In [0]:
# Pre-Crisis customer behavior

pre_crisis_df = (
    orders_df
    .filter(F.col("crisis_phase") == "Pre-Crisis")
    .groupBy("customer_id")
    .agg(
        F.count("order_id").alias("pre_order_count"),
        F.avg("net_revenue").alias("avg_order_value"),
        F.avg("delivery_delay_mins").alias("avg_delivery_delay")
    )
)

In [0]:
# Crisis Activity label creation

crisis_df = (
    orders_df
    .filter(F.col("crisis_phase") == "Crisis")
    .select("customer_id")
    .distinct()
    .withColumn("active_during_crisis", F.lit(1))
)


In [0]:
#create label column
ml_df = (
    pre_crisis_df
    .join(crisis_df, "customer_id", "left")
    .fillna({"active_during_crisis": 0})
    .withColumn("churn", F.when(F.col("active_during_crisis") == 0, 1).otherwise(0))
)


3: Add Ratings & Sentiment Features

In [0]:
ratings_df = (
    spark.table("uc_quickbite.silver_transform.ratings_enriched")
    .groupBy("customer_id")
    .agg(
        F.avg("rating").alias("avg_rating"),
        F.avg("sentiment_score").alias("avg_sentiment_score")
    )
)

ml_df = (
    ml_df
    .join(ratings_df, "customer_id", "left")
    .fillna(0)
)

4: Final Feature Selection

In [0]:
feature_cols = [
    "pre_order_count",
    "avg_order_value",
    "avg_delivery_delay",
    "avg_rating",
    "avg_sentiment_score"
]

5: Vector Assembler

In [0]:
assembler = VectorAssembler(
    inputCols=feature_cols,
    outputCol="features"
)

final_ml_df = assembler.transform(ml_df).select(
    "features",
    F.col("churn").cast(IntegerType()).alias("label")
)

6: Train / Test Split

In [0]:
train_df, test_df = final_ml_df.randomSplit([0.8, 0.2], seed=42)

7: Train Model with MLflow

In [0]:
final_ml_df.groupBy("label").count().show()

In [0]:
print("Train count:", train_df.count())
print("Test count:", test_df.count())


In [0]:
with mlflow.start_run():

    lr = LogisticRegression(
        featuresCol="features",
        labelCol="label",
        maxIter=20
    )

    model = lr.fit(train_df)
    predictions = model.transform(test_df)

    evaluator = BinaryClassificationEvaluator()
    auc = evaluator.evaluate(predictions)

    if auc is not None:
        mlflow.log_metric("AUC", float(auc))
    else:
        print("AUC is None – skipping metric logging")


In [0]:
print("AUC value:", auc)
print("AUC type:", type(auc))


In [0]:
import mlflow

mlflow.log_param("note", "Spark model logging skipped due to serverless issue")


In [0]:
%sql
CREATE SCHEMA IF NOT EXISTS uc_quickbite.ml;

In [0]:
%sql
CREATE VOLUME IF NOT EXISTS uc_quickbite.ml.volumes;
